# 05 — Class-imbalance handling

Compare SMOTE oversampling vs class weighting. On the positive class we optimize **PR-AUC / F1**. Weighting won here, so it is adopted.

In [ ]:
from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier
from src.models.preprocessing import build_preprocessor
from src.evaluation.metrics import evaluate_model
from src.models.model_zoo import build_zoo

Xtr, Xva, Xte, ytr, yva, yte = load_split_data()
pre = build_preprocessor(); pre.fit(Xtr, ytr)

# weighted model
xgb_w = build_zoo()["XGBoost"]
xgb_w.fit(pre.transform(Xtr), ytr)
_, rw = evaluate_model(xgb_w, pre, Xva, yva, "XGBoost")

# SMOTE model
Xsm, ysm = SMOTE(random_state=42).fit_resample(pre.transform(Xtr), ytr)
xgb_s = XGBClassifier(random_state=42, scale_pos_weight=10.0)
xgb_s.fit(Xsm, ysm)
_, rs = evaluate_model(xgb_s, pre, Xva, yva, "XGBoost-SMOTE")

print(f"Weighted  F1={rw["f1"]:.4f} ROC={rw["roc_auc"]:.4f} PR={rw["pr_auc"]:.4f}")
print(f"SMOTE     F1={rs["f1"]:.4f} ROC={rs["roc_auc"]:.4f} PR={rs["pr_auc"]:.4f}")